In [ ]:
from __future__ import annotations

import os
import pickle
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from typing import List, Dict, TypedDict

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import StateGraph, START, END

In [ ]:
llm = ChatOllama(model="deepseek-v3.1:671b-cloud")
embedder = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
class Node:
    def __init__(self, text, children=None):
        self.text = text
        self.children = children or []
        self.embedding = embedder.embed_query(text)

In [ ]:
def build_raptor_tree(pdf_path, save_path="raptor_tree.pkl"):
    print("📥 Loading PDF...")
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=5000,
        chunk_overlap=500
    )

    chunks = splitter.split_documents(docs)

    # STEP 1: LEAF NODES
    nodes = [Node(chunk.page_content) for chunk in chunks]

    print("🌳 Building RAPTOR Tree...")

    level = 0

    # Levels must remain sequential because each level depends on the previous one.
    while len(nodes) > 10:
        print(f"🔁 Level {level} clustering...")

        embeddings = np.array([n.embedding for n in nodes])

        # SIMPLE CLUSTERING (instead of GMM for simplicity)
        k = max(2, len(nodes)//5)

        from sklearn.cluster import KMeans
        kmeans = KMeans(n_clusters=k, random_state=0).fit(embeddings)

        clusters = {}
        for idx, label in enumerate(kmeans.labels_):
            clusters.setdefault(label, []).append(nodes[idx])

        cluster_groups = list(clusters.values())

        def summarize_cluster(cluster_nodes):
            combined_text = "\n".join([n.text for n in cluster_nodes])
            summary = llm.invoke(f"""
            Summarize the following content:

            {combined_text}

            Make it concise but informative.
            """).content
            return Node(summary, children=cluster_nodes)

        # Parallelize independent cluster summaries within the same level.
        max_workers = min(8, len(cluster_groups)) if cluster_groups else 1
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            new_nodes = list(executor.map(summarize_cluster, cluster_groups))

        nodes = new_nodes
        level += 1

    # ROOT
    root_nodes = nodes

    with open(save_path, "wb") as f:
        pickle.dump(root_nodes, f)

    print("✅ RAPTOR tree saved")

In [ ]:
def load_tree(path="raptor_tree.pkl"):
    if not os.path.exists(path):
        raise ValueError("❌ Tree not built yet")

    with open(path, "rb") as f:
        tree = pickle.load(f)

    print("✅ Tree loaded (no rebuild)")
    return tree

In [ ]:
def flatten_tree(nodes):
    all_nodes = []

    def dfs(node):
        all_nodes.append(node)
        for child in node.children:
            dfs(child)

    for n in nodes:
        dfs(n)

    return all_nodes

In [ ]:
class RaptorState(TypedDict, total=False):
    query: str
    all_nodes: List[Node]
    context: str
    answer: str

In [ ]:
def retrieve(state: RaptorState):
    query = state.get("query", "")
    nodes = state.get("all_nodes", [])

    query_emb = embedder.embed_query(query)

    scores = []
    for node in nodes:
        sim = np.dot(query_emb, node.embedding) / (
            np.linalg.norm(query_emb) * np.linalg.norm(node.embedding)
        )
        scores.append(sim)

    # top-k nodes
    top_indices = np.argsort(scores)[-10:]
    selected = [nodes[i].text for i in top_indices]

    context = "\n\n".join(selected)

    return {"context": context}

In [ ]:
def generate(state: RaptorState):
    answer = llm.invoke(f"""
    Answer the question using the context.

    QUESTION:
    {state.get('query', '')}

    CONTEXT:
    {state.get('context', '')}
    """).content

    return {"answer": answer}

In [ ]:
def build_pipeline():
    builder = StateGraph(RaptorState)

    builder.add_node("retrieve", retrieve)
    builder.add_node("generate", generate)

    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)

    return builder.compile()

In [ ]:
app = build_pipeline()
app

In [12]:
if __name__ == "__main__":
    pdf_path = r"D:\Capstone\capstone_proposal.pdf"

    # =========================
    # BUILD TREE ONCE
    # =========================
    if not os.path.exists("raptor_tree.pkl"):
        build_raptor_tree(pdf_path)

    # =========================
    # LOAD TREE
    # =========================
    tree = load_tree()

    # FLATTEN TREE (Collapsed Tree Retrieval)
    all_nodes = flatten_tree(tree)

    app = build_pipeline()

    print("\n🌳 RAPTOR Ready (collapsed retrieval)\n")

    while True:
        query = input("👉 Ask: ")

        if query.lower() == "exit":
            break

        result = app.invoke({
            "query": query,
            "all_nodes": all_nodes
        })

        print("\n✅ ANSWER:\n")
        print(result["answer"])
        print("\n" + "="*60)


✅ ANSWER:

Based on the provided context, there is one Muslim person involved in the project: **Sajid Miya (102367013)**.

His role is **System Architecture & AI Development Lead**. His responsibilities include:
*   Designing the overall system architecture and modular framework.
*   Leading AI development, including multi-agent orchestration and Hybrid RAG implementation.
*   Conducting AI research and model integration.
*   Contributing to backend development and desktop application support.

